# End-sem 06. Explanations, case studies and the dashboard screenshots
SHAP on the ranker, the translated radar, three hits and one bad miss. The same figures are drawn live by `app/shadow_squad.py`.

In [1]:
import sys, os, subprocess, time; sys.path.insert(0, 'src')
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt, shap
from common import *
import engine as E, viz
lake = E.Lake(); models = E.load_models(); PF = pd.read_parquet('lake/pairs.parquet'); LG = pd.read_parquet('lake/scores_lgbm_v2.parquet'); EV = pd.read_parquet('lake/replacement_events')
EV['qid'] = EV['C'].astype(str) + '_' + EV['t'].astype(str) + '_' + EV['S'].astype(str); M = {}

/home/srioo/Desktop/Projects/root/BigP/bigdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. What drives the ranking overall (SHAP, out-of-time rows only)

In [2]:
parts = []
for Y in (2020, 2021, 2022, 2023, 2024):
    X = PF[(PF['t'] == Y) & PF['q_in_pool']]; parts.append((viz.explain(models[Y], X[E.FEATS]).values, X))
SV = np.vstack([p[0] for p in parts]); XA = pd.concat([p[1] for p in parts]); imp = pd.Series(np.abs(SV).mean(0), index=E.FEATS).sort_values()
dirn = {f: float(np.corrcoef(XA[f].fillna(XA[f].median()), SV[:, i])[0, 1]) for i, f in enumerate(E.FEATS)}
fig, ax = plt.subplots(figsize=(6.6, 3.2)); ax.grid(axis='y', visible=False); ax.barh([viz.NICE[f] for f in imp.index], imp.values, color=[ORANGE if dirn[f] > 0 else BLUE for f in imp.index]); ax.set_xlabel('mean |SHAP|, unseen events (orange raises rank, blue lowers)'); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.savefig('outputs/figs/p6_shap_global.png'); plt.close()
M['shap_global'] = {'mean_abs_shap': imp.round(4).to_dict(), 'direction_corr': {k: round(v, 3) for k, v in dirn.items()}, 'rows': len(XA)}; print(M['shap_global']['mean_abs_shap'])

{'pipeline_rank': 0.0185, 'pattern_match': 0.0187, 'composite_perf_index': 0.0217, 'club_taste': 0.0274, 'traj_sim': 0.0281, 'sim_translated': 0.0286, 'hazard': 0.0469, 'age_num': 0.0485, 'log_value': 0.0529, 'value_gap': 0.3572}


## 2. Case studies: three hits and one miss
Chosen by rule, not by hand: the three unseen events with the best rank for the real signing (ties broken by the larger fee), and the unseen event where a real signing was ranked worst.

In [3]:
R = LG.merge(PF[['qid', 'player_id', 'candidate']], on=['qid', 'player_id']); R['rank'] = R.groupby('qid')['score'].rank(ascending=False, method='first'); R['pool'] = R.groupby('qid')['score'].transform('size')
tr = R[R['y'] == 1].merge(EV[['qid', 'N', 'club', 't', 'S_name', 'P', 'N_name', 'N_from', 'fee', 'C', 'S', 'league']].rename(columns={'N': 'player_id', 't': 't_'}), on=['qid', 'player_id'])
hits = tr[tr['rank'] <= 10].sort_values(['rank', 'fee'], ascending=[True, False]).drop_duplicates('qid').head(3); miss = tr.sort_values('rank', ascending=False).head(1); cases = []
for kind, row in [('hit', r) for r in hits.itertuples()] + [('miss', r) for r in miss.itertuples()]:
    X = E.rank(lake, models, row.C, row.t, row.S, row.P, top=10 ** 6); X['rank'] = range(1, len(X) + 1); me = X[X['player_id'] == row.player_id].iloc[0]; tag = f"{kind}_{row.club.replace(' ', '')}_{row.t}_{row.N_name.split()[-1]}"
    pct = {f: float((X[f] < me[f]).mean()) for f in E.FEATS}
    for who, lab_ in ((me, 'signed'), (X.iloc[0], 'top1')):
        viz.radar(lake, who, row.S, row.t).savefig(f'outputs/figs/case_{tag}_{lab_}_radar.png'); plt.close('all'); viz.waterfall(models[row.t], X, int(who.name)).savefig(f'outputs/figs/case_{tag}_{lab_}_shap.png'); plt.close('all')
    cases.append({'kind': kind, 'tag': tag, 'club': row.club, 'league': row.league, 'season': int(row.t), 'starter': row.S_name, 'position': row.P, 'signed': row.N_name, 'from': row.N_from, 'fee': None if pd.isna(row.fee) else float(row.fee), 'rank': int(me['rank']), 'pool': len(X),
                  'signed_feature_percentile_in_pool': {k: round(v, 2) for k, v in pct.items()}, 'signed_values': {f: (None if pd.isna(me[f]) else round(float(me[f]), 3)) for f in E.FEATS}, 'starter_value': None if pd.isna(me['S_value']) else float(me['S_value']), 'signed_value': float(me['value']),
                  'top5': X.head(5)[['player', 'squad', 'age_num', 'value', 'score']].round(3).to_dict('records'), 'C': int(row.C), 'S': int(row.S), 'N': int(row.player_id), 'rank_lo': None})
    print(f"{kind.upper()}: {row.club} {row.t}-{str(row.t + 1)[2:]}, {row.S_name} ({row.P}) -> {row.N_name} from {row.N_from}; ranked {int(me['rank'])} of {len(X)}; percentiles {cases[-1]['signed_feature_percentile_in_pool']}")
M['cases'] = cases

HIT: Manchester City 2024-25, Rico Lewis (FB) -> Rayan Aït-Nouri from Wolves; ranked 1 of 103; percentiles {'sim_translated': 0.39, 'traj_sim': 0.2, 'pipeline_rank': 0.4, 'club_taste': 0.33, 'hazard': 0.99, 'log_value': 0.96, 'value_gap': 0.01, 'age_num': 0.27, 'composite_perf_index': 0.91, 'pattern_match': 0.98}


HIT: Rennes 2022-23, Flavien Tait (CM) -> Enzo Le Fée from FC Lorient; ranked 1 of 89; percentiles {'sim_translated': 0.91, 'traj_sim': 0.92, 'pipeline_rank': 0.74, 'club_taste': 0.47, 'hazard': 0.98, 'log_value': 0.74, 'value_gap': 0.55, 'age_num': 0.16, 'composite_perf_index': 0.99, 'pattern_match': 0.99}


HIT: Real Sociedad 2021-22, Adnan Januzaj (W) -> Takefusa Kubo from Real Madrid; ranked 1 of 87; percentiles {'sim_translated': 0.99, 'traj_sim': 0.99, 'pipeline_rank': 0.18, 'club_taste': 0.32, 'hazard': 0.9, 'log_value': 0.46, 'value_gap': 0.17, 'age_num': 0.09, 'composite_perf_index': 0.9, 'pattern_match': 0.66}


MISS: Chelsea 2023-24, Thiago Silva (CB) -> Tosin Adarabioyo from Fulham; ranked 127 of 141; percentiles {'sim_translated': 0.84, 'traj_sim': 0.85, 'pipeline_rank': 0.19, 'club_taste': 0.28, 'hazard': 0.38, 'log_value': 0.87, 'value_gap': 0.87, 'age_num': 0.67, 'composite_perf_index': 0.6, 'pattern_match': 0.92}


## 3. Dashboard screenshots (taken from the running Streamlit app)

In [4]:
import json
sq = pd.read_parquet('lake/live_squads.parquet'); lr = sq[sq['age_today'] < 30].nlargest(1, 'risk').iloc[0]
jobs = [{'tag': c['tag'], 'season': c['season'], 'club': c['club'], 'starter': c['S'], 'cand': c['N'], 'kind': c['kind']} for c in cases] + [{'tag': 'live', 'season': 'live', 'club': lr['club'], 'starter': int(lr['player_id']), 'cand': None, 'kind': 'live'}]
json.dump(jobs, open('outputs/screens/cases.json', 'w')); M['live_example'] = {'club': lr['club'], 'starter': lr['name'], 'risk': float(lr['risk']), 'position': lr['fine_pos']}
out = subprocess.run([sys.executable, 'src/screens.py', 'outputs/screens/cases.json'], capture_output=True, text=True); print(out.stderr[-800:]); shots = out.stdout.split()
assert len(shots) >= 3 * len(cases), 'screenshots missing'
M['screens'] = shots; print(shots); _ = put_metrics('phase6', M)


['outputs/screens/hit_ManchesterCity_2024_Aït-Nouri_squad.png', 'outputs/screens/hit_ManchesterCity_2024_Aït-Nouri_cards.png', 'outputs/screens/hit_ManchesterCity_2024_Aït-Nouri_compare.png', 'outputs/screens/hit_Rennes_2022_Fée_squad.png', 'outputs/screens/hit_Rennes_2022_Fée_cards.png', 'outputs/screens/hit_Rennes_2022_Fée_compare.png', 'outputs/screens/hit_RealSociedad_2021_Kubo_squad.png', 'outputs/screens/hit_RealSociedad_2021_Kubo_cards.png', 'outputs/screens/hit_RealSociedad_2021_Kubo_compare.png', 'outputs/screens/miss_Chelsea_2023_Adarabioyo_squad.png', 'outputs/screens/miss_Chelsea_2023_Adarabioyo_cards.png', 'outputs/screens/live_squad.png', 'outputs/screens/live_cards.png', 'outputs/screens/live_compare.png', 'outputs/screens/live_rebuild.png']
